# Phase 3B: Transfer Entropy - Non-Linear Information Flow

## Objective

Quantify the **directional, non-linear information flow** from news sentiment to stock returns that linear Granger Causality cannot detect.

## Core Hypothesis

**High Beta stocks** act as **"Information Sinks"** —> they absorb news sentiment at a significantly higher rate than **Low Beta stocks**, which are driven by fundamentals rather than sentiment.

## Transfer Entropy vs Granger Causality

| Aspect | Granger Causality | Transfer Entropy |
|--------|------------------|------------------|
| **Type** | Linear regression | Information theory |
| **Detects** | Linear predictive relationships | Non-linear information flow |
| **Assumption** | Gaussian errors, linear dynamics | None (model-free) |
| **Captures** | Smooth, gradual effects | Abrupt shocks, regime changes |

---

## Mathematical Foundation

**Transfer Entropy** (Schreiber, 2000):

$$TE_{X \to Y} = H(Y_t | Y_{t-1}) - H(Y_t | Y_{t-1}, X_{t-1})$$

Where:
- $H(Y_t | Y_{t-1})$: Entropy of future returns given past returns (baseline uncertainty)
- $H(Y_t | Y_{t-1}, X_{t-1})$: Entropy of future returns given past returns AND past sentiment
- **Interpretation**: How many bits of information does knowing yesterday's sentiment provide about today's return?

**Directionality**:
- **Forward TE** ($Sentiment \to Price$): News drives price
- **Backward TE** ($Price \to Sentiment$): Price influences news (reflexivity)
- **Net TE** = Forward - Backward: Net information flow

---

## 1. Setup: Imports and Paths

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import entropy, pearsonr
from scipy.special import rel_entr

import matplotlib.pyplot as plt
import seaborn as sns

import os
import random
from tqdm import tqdm
from itertools import product

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.6f}'.format)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

np.random.seed(42)
random.seed(42)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
project_root = '/content/drive/MyDrive/market-sentiment-impact-analysis'
data_processed = os.path.join(project_root, 'data', 'processed')
plots = os.path.join(project_root, 'plots')

print(f'Project Root: {project_root}')
print(f'Processed Data: {data_processed}')
print(f'Plots: {plots}')

Project Root: /content/drive/MyDrive/market-sentiment-impact-analysis
Processed Data: /content/drive/MyDrive/market-sentiment-impact-analysis/data/processed
Plots: /content/drive/MyDrive/market-sentiment-impact-analysis/plots


In [4]:
N_STATES = 3          # 0=Bearish/Negative, 1=Neutral, 2=Bullish/Positive
LAG_K = 1
N_SURROGATES = 200    # num of shuffles for significance testing
ALPHA = 0.05
MIN_OBS = 100

print('Transfer Entropy Configuration:')
print(f'  States: {N_STATES} (Discrete market regimes)')
print(f'  Lag: {LAG_K} trading day(s)')
print(f'  Surrogate tests: {N_SURROGATES}')
print(f'  Significance: α = {ALPHA}')
print(f'  Min observations: {MIN_OBS}')

Transfer Entropy Configuration:
  States: 3 (Discrete market regimes)
  Lag: 1 trading day(s)
  Surrogate tests: 200
  Significance: α = 0.05
  Min observations: 100


---

## 2. Data Preparation & State Encoding

### 2.1 Load Data

In [5]:
master_df = pd.read_csv(os.path.join(data_processed, 'master_df.csv'))
master_df['Date'] = pd.to_datetime(master_df['Date'])
master_df.head(3)

,Date,Ticker,Log_Return,Sector,Beta,Beta_Group,vader_mean,finbert_mean,article_count,vader_std,finbert_std,market_vader_mean,market_finbert_mean,market_article_count,market_vader_std,market_finbert_std
0,2018-01-03,AEP,-0.008460,Utilities,0.585551,Low Beta,0.000000,0.000000,0.000000,0.000000,0.000000,0.065113,0.117741,15.000000,0.343850,0.598781
1,2018-01-03,AMP,-0.004956,Financials,1.770886,High Beta,0.000000,0.000000,0.000000,0.000000,0.000000,0.065113,0.117741,15.000000,0.343850,0.598781
2,2018-01-03,APA,0.022985,Energy,1.950480,High Beta,0.000000,0.000000,0.000000,0.000000,0.000000,0.065113,0.117741,15.000000,0.343850,0.598781


### 2.2 Max-Entropy Discretization

**Why Discretization?**
- Transfer entropy requires **discrete states** (not continuous values)
- We bin continuous data into 3 states: {Bearish/Negative, Neutral, Bullish/Positive}

**Why Equal-Frequency (Quantile) Binning?**
- Ensures each state has ~33% probability → maximum entropy
- Each ticker gets its own bins (personalized thresholds)

In [6]:
def discretize_ticker_data(ticker_df, n_states=3):
    """
    Discretize returns and sentiment into equal-frequency bins per ticker.

    Returns:
        ticker_df with new columns: return_state, vader_state, finbert_state
    """
    df = ticker_df.copy()

    try:
        df['return_state'] = pd.qcut(
            df['Log_Return'],
            q=n_states,
            labels=False,
            duplicates='drop'
        )
    except ValueError:
        df['return_state'] = 1

    try:
        df['vader_state'] = pd.qcut(
            df['vader_mean'],
            q=n_states,
            labels=False,
            duplicates='drop'
        )
    except ValueError:
        df['vader_state'] = 1

    try:
        df['finbert_state'] = pd.qcut(
            df['finbert_mean'],
            q=n_states,
            labels=False,
            duplicates='drop'
        )
    except ValueError:
        df['finbert_state'] = 1

    return df

print('Discretizing data per ticker...')
master_df = master_df.groupby('Ticker', group_keys=False).apply(
    lambda x: discretize_ticker_data(x, n_states=N_STATES)
)

print('\n  Discretization complete')
print(f'\nState distribution (all tickers combined):')
print(f"  Return states: {master_df['return_state'].value_counts().sort_index().to_dict()}")
print(f"  VADER states: {master_df['vader_state'].value_counts().sort_index().to_dict()}")
print(f"  FinBERT states: {master_df['finbert_state'].value_counts().sort_index().to_dict()}")

Discretizing data per ticker...

  Discretization complete

State distribution (all tickers combined):
  Return states: {0: 13075, 1: 12698, 2: 12567}
  VADER states: {0: 16873, 1: 9332, 2: 12135}
  FinBERT states: {0: 15403, 1: 11159, 2: 11778}


In [7]:
sample_ticker = master_df['Ticker'].iloc[0]
sample_df = master_df[master_df['Ticker'] == sample_ticker].copy()

print(f'Discretization validation for {sample_ticker}:')
print(f'\nOriginal vs Discretized Returns:')
print(sample_df[['Date', 'Log_Return', 'return_state']].head(10))

print(f'\nState transition check (should have ~equal frequencies):')
print(f"  Return states: {sample_df['return_state'].value_counts(normalize=True).sort_index()}")
print(f"  VADER states: {sample_df['vader_state'].value_counts(normalize=True).sort_index()}")

Discretization validation for AEP:

Original vs Discretized Returns:
          Date  Log_Return  return_state
0   2018-01-03   -0.008460             0
60  2018-01-04   -0.011909             0
120 2018-01-05   -0.002116             1
180 2018-01-08    0.008719             2
240 2018-01-09   -0.011831             0
300 2018-01-10   -0.015420             0
360 2018-01-11   -0.011141             0
420 2018-01-12   -0.018651             0
480 2018-01-16    0.000593             1
540 2018-01-17    0.011050             2

State transition check (should have ~equal frequencies):
  Return states: return_state
0   0.333333
1   0.333333
2   0.333333
Name: proportion, dtype: float64
  VADER states: vader_state
0   0.378717
1   0.287950
2   0.333333
Name: proportion, dtype: float64
